In [1]:
import os
import pandas as pd
from pathlib import Path
PROCESSED_FILE = Path(r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed_sample.csv")

processed = pd.read_csv(PROCESSED_FILE)

print(processed.shape)
print(processed.head(10))

(3368, 14)
  user_id      timestamp  solving_id question_id user_answer correct_answer  \
0      u1  1565096190868           1       q5012           b              c   
1      u1  1565096221062           2       q4706           c              c   
2      u1  1565096293432           3       q4366           b              b   
3      u1  1565096339668           4       q4829           a              c   
4      u1  1565096401774           5       q6528           b              d   
5      u1  1565096463370           6       q4793           a              b   
6      u1  1565096501746           7       q6488           a              a   
7      u1  1565097101361           8        q356           b              b   
8      u1  1565097171393           9       q1382           c              c   
9      u1  1565097240758          10        q830           b              b   

   correctness              tags  elapsed_time  elapsed_seconds  \
0            0                74         38000      

In [2]:
# ============================================================
# FEATURE 1: QUESTION ID → 64-D EMBEDDING
# ============================================================

import torch
import torch.nn as nn

# Create question vocabulary
unique_questions = processed["question_id"].unique()
num_questions = len(unique_questions)

# Question ID → embedding index
question_to_index = {
    question_id: index
    for index, question_id in enumerate(unique_questions)
}

# Add lookup index to dataframe
processed["question_index"] = (
    processed["question_id"].map(question_to_index)
)

# Validate mapping
if processed["question_index"].isna().any():
    raise ValueError("Some question IDs could not be mapped.")

# Create 64-D trainable embedding
QUESTION_EMBEDDING_DIM = 64

question_embedding = nn.Embedding(
    num_embeddings=num_questions,
    embedding_dim=QUESTION_EMBEDDING_DIM
)

# Convert indices to tensor
question_indices = torch.tensor(
    processed["question_index"].to_numpy(),
    dtype=torch.long
)

# Generate vectors
question_vectors = question_embedding(question_indices)

# Sanity checks
assert question_vectors.shape == (
    len(processed),
    64
)

print("Unique questions:", num_questions)
print("Question vectors:", question_vectors.shape)
print("\nFirst question:")
print("ID:", processed.iloc[0]["question_id"])
print("Index:", processed.iloc[0]["question_index"])
print("64-D vector:")
print(question_vectors[0])

Unique questions: 2320
Question vectors: torch.Size([3368, 64])

First question:
ID: q5012
Index: 0
64-D vector:
tensor([-0.2151, -0.7868, -0.9192,  0.1661, -0.3877, -0.7923,  0.4576, -2.1967,
        -0.6014, -0.3619,  1.5995, -0.3520,  1.2397, -0.5095,  0.8662,  1.4772,
         1.0243, -0.0989,  1.2566, -0.1003, -0.9633,  1.1506,  0.6339, -0.5129,
        -1.0365,  2.1374,  1.3713,  0.4153,  0.8220, -1.4969,  1.1647,  0.1182,
         0.2922,  0.7909,  0.9594, -1.1012, -1.5508,  0.2995,  0.3042,  1.0411,
        -1.3258, -1.1839, -0.4167, -0.3193, -1.1641,  1.0390,  0.6589,  2.0123,
         0.9948,  0.5371,  0.6665,  1.1573,  0.4674,  1.8396,  0.7028, -0.3300,
        -0.4932, -0.2883,  0.9977, -0.6191, -0.6562, -1.0189, -0.1291,  0.2268],
       grad_fn=<SelectBackward0>)


In [3]:
# ============================================================
# FEATURE 2: SKILL TAGS → 32-D REPRESENTATION
# ============================================================

# Extract unique skill tags
all_tags = (
    processed["tags"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

unique_skills = sorted(all_tags.unique())
num_skills = len(unique_skills)

# Skill ID → embedding index
skill_to_index = {
    skill_id: index
    for index, skill_id in enumerate(unique_skills)
}

# Create 32-D trainable skill embedding
SKILL_EMBEDDING_DIM = 32

skill_embedding = nn.Embedding(
    num_embeddings=num_skills,
    embedding_dim=SKILL_EMBEDDING_DIM
)

# Function to convert one interaction's tags → 32-D
def get_skill_vector(tag_string):

    tags = [
        tag.strip()
        for tag in str(tag_string).split(";")
        if tag.strip()
    ]

    indices = [
        skill_to_index[tag]
        for tag in tags
    ]

    indices_tensor = torch.tensor(
        indices,
        dtype=torch.long
    )

    # Individual skill vectors
    embeddings = skill_embedding(indices_tensor)

    # Multiple skills → one 32-D vector
    return embeddings.mean(dim=0)


# Generate skill vectors for every interaction
skill_vectors = torch.stack(
    processed["tags"]
    .apply(get_skill_vector)
    .tolist()
)

# Sanity checks
assert skill_vectors.shape == (
    len(processed),
    32
)

print("Unique skills:", num_skills)
print("Skill vectors:", skill_vectors.shape)

print("\nFirst interaction:")
print("Question:", processed.iloc[0]["question_id"])
print("Tags:", processed.iloc[0]["tags"])

print("\n32-D skill vector:")
print(skill_vectors[0])

Unique skills: 183
Skill vectors: torch.Size([3368, 32])

First interaction:
Question: q5012
Tags: 74

32-D skill vector:
tensor([-0.5551,  0.8207,  0.0949,  0.0620, -0.1993, -1.0810,  1.3013,  0.9624,
        -0.1692, -0.1393, -1.6890,  0.9569,  1.5653, -0.7870, -1.3288,  1.3646,
         0.6919,  0.7365,  0.3480,  0.5127, -0.7910,  0.2721,  0.4511, -1.2101,
        -0.3423, -0.8428, -0.6824, -1.3253,  0.7867,  2.1305, -0.1864,  0.9451],
       grad_fn=<SelectBackward0>)


In [4]:
# ============================================================
# FEATURE 3: CORRECTNESS → 8-D EMBEDDING
# ============================================================

# Validate correctness
unique_correctness = sorted(
    processed["correctness"].dropna().unique()
)

if not set(unique_correctness).issubset({0, 1}):
    raise ValueError(
        "Correctness must contain only 0 and 1."
    )

# Create 8-D trainable embedding
CORRECTNESS_EMBEDDING_DIM = 8

correctness_embedding = nn.Embedding(
    num_embeddings=2,
    embedding_dim=CORRECTNESS_EMBEDDING_DIM
)

# Convert correctness to tensor
correctness_indices = torch.tensor(
    processed["correctness"].to_numpy(),
    dtype=torch.long
)

# Generate vectors
correctness_vectors = correctness_embedding(
    correctness_indices
)

# Sanity checks
assert correctness_vectors.shape == (
    len(processed),
    8
)

print("Correctness values:", unique_correctness)
print("Correctness vectors:", correctness_vectors.shape)

print("\nFirst interaction:")
print("Correctness:", processed.iloc[0]["correctness"])
print("8-D vector:")
print(correctness_vectors[0])

Correctness values: [np.int64(0), np.int64(1)]
Correctness vectors: torch.Size([3368, 8])

First interaction:
Correctness: 0
8-D vector:
tensor([-0.6448,  1.4522,  0.4191, -0.5371,  2.1081, -0.6916,  0.7161, -0.7133],
       grad_fn=<SelectBackward0>)


In [5]:
correct_vector = correctness_embedding(
    torch.tensor([1])
)[0]

incorrect_vector = correctness_embedding(
    torch.tensor([0])
)[0]

print("Correct vector:")
print(correct_vector)

print("\nIncorrect vector:")
print(incorrect_vector)

Correct vector:
tensor([ 0.4830, -2.0777, -1.8334, -0.1725,  0.6813,  0.2409,  0.5067,  0.4111],
       grad_fn=<SelectBackward0>)

Incorrect vector:
tensor([-0.6448,  1.4522,  0.4191, -0.5371,  2.1081, -0.6916,  0.7161, -0.7133],
       grad_fn=<SelectBackward0>)


In [6]:
# ============================================================
# FEATURE 4: ELAPSED TIME → 16-D REPRESENTATION
# ============================================================

# ------------------------------------------------------------
# 1. Use the already processed log elapsed time
# ------------------------------------------------------------

log_elapsed_time = processed["log_elapsed_time"]

# ------------------------------------------------------------
# 2. Normalize
#
# For the current prototype only:
# statistics come from this user's data.
#
# Final implementation:
# calculate these ONLY from training data.
# ------------------------------------------------------------

elapsed_mean = log_elapsed_time.mean()
elapsed_std = log_elapsed_time.std()

if elapsed_std == 0:
    raise ValueError(
        "Elapsed-time standard deviation is zero."
    )

normalized_elapsed_time = (
    (log_elapsed_time - elapsed_mean)
    / elapsed_std
)

# ------------------------------------------------------------
# 3. Convert normalized scalar to tensor
# ------------------------------------------------------------

elapsed_time_tensor = torch.tensor(
    normalized_elapsed_time.to_numpy(),
    dtype=torch.float32
).unsqueeze(1)

# ------------------------------------------------------------
# 4. Learnable 1-D → 16-D projection
# ------------------------------------------------------------

TIME_EMBEDDING_DIM = 16

elapsed_time_projection = nn.Linear(
    in_features=1,
    out_features=TIME_EMBEDDING_DIM
)

# ------------------------------------------------------------
# 5. Generate 16-D vectors
# ------------------------------------------------------------

elapsed_time_vectors = elapsed_time_projection(
    elapsed_time_tensor
)

# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

assert elapsed_time_vectors.shape == (
    len(processed),
    16
)

assert torch.isfinite(
    elapsed_time_vectors
).all()

print("Elapsed-time mean:", elapsed_mean)
print("Elapsed-time std:", elapsed_std)

print(
    "Elapsed-time vectors:",
    elapsed_time_vectors.shape
)

print("\nFirst interaction:")
print(
    "Log elapsed time:",
    processed.iloc[0]["log_elapsed_time"]
)

print(
    "Normalized:",
    normalized_elapsed_time.iloc[0]
)

print("\n16-D vector:")
print(elapsed_time_vectors[0])

Elapsed-time mean: 3.4765421310437405
Elapsed-time std: 0.7024972006936938
Elapsed-time vectors: torch.Size([3368, 16])

First interaction:
Log elapsed time: 3.6635616461296463
Normalized: 0.2662210111317595

16-D vector:
tensor([ 0.1877, -0.6334, -0.4795,  0.1328,  0.0412,  0.6707, -0.4374,  0.5531,
        -0.6462, -0.6680,  0.5728,  0.5112,  0.1600,  0.8358,  0.2202,  0.6429],
       grad_fn=<SelectBackward0>)


In [7]:
# ============================================================
# FEATURE 5: ATTEMPT INDEX → 8-D EMBEDDING
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Read the already-processed attempt_index
# ------------------------------------------------------------

if "attempt_index" not in processed.columns:
    raise ValueError(
        "'attempt_index' is missing from processed data."
    )

if processed["attempt_index"].isna().any():
    raise ValueError(
        "'attempt_index' contains missing values."
    )

# Attempt index starts at 1 in our processed dataset
min_attempt = int(processed["attempt_index"].min())
max_attempt = int(processed["attempt_index"].max())

print("Minimum attempt:", min_attempt)
print("Maximum attempt:", max_attempt)

# ------------------------------------------------------------
# 2. Convert 1-based attempt numbers to 0-based
#    embedding lookup indices
#
#    attempt 1 → index 0
#    attempt 2 → index 1
#    attempt 3 → index 2
# ------------------------------------------------------------

attempt_indices = torch.tensor(
    processed["attempt_index"].to_numpy() - 1,
    dtype=torch.long
)

# ------------------------------------------------------------
# 3. Create the 8-D trainable embedding
# ------------------------------------------------------------

ATTEMPT_EMBEDDING_DIM = 8

attempt_embedding = nn.Embedding(
    num_embeddings=max_attempt,
    embedding_dim=ATTEMPT_EMBEDDING_DIM
)

# ------------------------------------------------------------
# 4. Generate the 8-D representation
# ------------------------------------------------------------

attempt_vectors = attempt_embedding(
    attempt_indices
)

# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

assert attempt_indices.shape == (len(processed),)

assert attempt_vectors.shape == (
    len(processed),
    8
)

assert torch.isfinite(attempt_vectors).all()

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\nAttempt embedding:")
print(attempt_embedding)

print("\nAttempt index tensor shape:")
print(attempt_indices.shape)

print("\nAttempt vector shape:")
print(attempt_vectors.shape)

print("\nFirst interaction:")
print("Question ID:", processed.iloc[0]["question_id"])
print("Attempt index:", processed.iloc[0]["attempt_index"])
print("Embedding index:", attempt_indices[0].item())

print("\n8-D attempt vector:")
print(attempt_vectors[0])

print("\n✓ Attempt-index representation successfully created.")
print(
    f"✓ {len(processed)} interactions → "
    f"8-D attempt vectors"
)

Minimum attempt: 1
Maximum attempt: 5

Attempt embedding:
Embedding(5, 8)

Attempt index tensor shape:
torch.Size([3368])

Attempt vector shape:
torch.Size([3368, 8])

First interaction:
Question ID: q5012
Attempt index: 1
Embedding index: 0

8-D attempt vector:
tensor([ 0.4255,  0.3107,  0.2726,  1.3462,  0.2131, -0.8205,  1.2915,  0.5318],
       grad_fn=<SelectBackward0>)

✓ Attempt-index representation successfully created.
✓ 3368 interactions → 8-D attempt vectors


In [8]:
# ============================================================
# FEATURE 6: TIME DELTA → 8-D REPRESENTATION
# ============================================================

# ------------------------------------------------------------
# 1. Validate the existing processed column
# ------------------------------------------------------------

if "log_time_delta" not in processed.columns:
    raise ValueError(
        "'log_time_delta' is missing from processed data."
    )

if processed["log_time_delta"].isna().any():
    raise ValueError(
        "'log_time_delta' contains missing values."
    )


# ------------------------------------------------------------
# 2. Read the already-processed log time delta
# ------------------------------------------------------------

log_time_delta = processed["log_time_delta"]


# ------------------------------------------------------------
# 3. Normalize
#
# For the current prototype:
# statistics are calculated from this user's data.
#
# Final implementation:
# calculate statistics ONLY from training data.
# ------------------------------------------------------------

delta_mean = log_time_delta.mean()
delta_std = log_time_delta.std()

if delta_std == 0:
    raise ValueError(
        "Time-delta standard deviation is zero."
    )

normalized_time_delta = (
    (log_time_delta - delta_mean)
    / delta_std
)


# ------------------------------------------------------------
# 4. Convert normalized values to PyTorch tensor
# ------------------------------------------------------------

time_delta_tensor = torch.tensor(
    normalized_time_delta.to_numpy(),
    dtype=torch.float32
).unsqueeze(1)


# ------------------------------------------------------------
# 5. Create learnable 1-D → 8-D projection
# ------------------------------------------------------------

TIME_DELTA_EMBEDDING_DIM = 8

time_delta_projection = nn.Linear(
    in_features=1,
    out_features=TIME_DELTA_EMBEDDING_DIM
)


# ------------------------------------------------------------
# 6. Generate 8-D time-delta representations
# ------------------------------------------------------------

time_delta_vectors = time_delta_projection(
    time_delta_tensor
)


# ------------------------------------------------------------
# 7. Sanity checks
# ------------------------------------------------------------

assert time_delta_tensor.shape == (
    len(processed),
    1
)

assert time_delta_vectors.shape == (
    len(processed),
    8
)

assert torch.isfinite(
    time_delta_vectors
).all()


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("=" * 60)
print("TIME DELTA REPRESENTATION")
print("=" * 60)

print("\nNormalization statistics:")
print(f"Mean: {delta_mean:.6f}")
print(f"Std : {delta_std:.6f}")

print("\nTensor shapes:")
print(
    "Input tensor:",
    time_delta_tensor.shape
)

print(
    "Output vectors:",
    time_delta_vectors.shape
)


# ------------------------------------------------------------
# 9. Display first interaction
# ------------------------------------------------------------

first = processed.iloc[0]

print("\nFirst interaction:")

print(
    "Time delta:",
    first["time_delta_seconds"],
    "seconds"
)

print(
    "Log time delta:",
    first["log_time_delta"]
)

print(
    "Normalized time delta:",
    normalized_time_delta.iloc[0]
)

print("\n8-D time-delta vector:")
print(time_delta_vectors[0])


# ------------------------------------------------------------
# 10. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ Time-delta representation successfully created.")
print(
    f"✓ {len(processed)} interactions → "
    f"8-D time-delta vectors"
)
print("=" * 60)

TIME DELTA REPRESENTATION

Normalization statistics:
Mean: 3.502407
Std : 2.810031

Tensor shapes:
Input tensor: torch.Size([3368, 1])
Output vectors: torch.Size([3368, 8])

First interaction:
Time delta: 0.0 seconds
Log time delta: 0.0
Normalized time delta: -1.2463945916425794

8-D time-delta vector:
tensor([-0.6113, -0.4888, -1.1182,  0.2268, -2.0104, -1.2761,  1.8275,  1.9237],
       grad_fn=<SelectBackward0>)

✓ Time-delta representation successfully created.
✓ 3368 interactions → 8-D time-delta vectors


In [9]:
# ============================================================
# COMBINE ALL FEATURE REPRESENTATIONS → 136-D INTERACTION
# ============================================================

import torch


# ------------------------------------------------------------
# 1. Check that all six representations exist
# ------------------------------------------------------------

required_vectors = {
    "question_vectors": question_vectors,
    "skill_vectors": skill_vectors,
    "correctness_vectors": correctness_vectors,
    "elapsed_time_vectors": elapsed_time_vectors,
    "attempt_vectors": attempt_vectors,
    "time_delta_vectors": time_delta_vectors,
}

for name, tensor in required_vectors.items():
    if not isinstance(tensor, torch.Tensor):
        raise TypeError(
            f"{name} is not a PyTorch tensor."
        )

    if tensor.shape[0] != len(processed):
        raise ValueError(
            f"{name} has {tensor.shape[0]} interactions, "
            f"but processed has {len(processed)} rows."
        )


# ------------------------------------------------------------
# 2. Print individual dimensions
# ------------------------------------------------------------

print("Individual feature shapes:")
print("Question      :", question_vectors.shape)
print("Skill         :", skill_vectors.shape)
print("Correctness   :", correctness_vectors.shape)
print("Elapsed time  :", elapsed_time_vectors.shape)
print("Attempt       :", attempt_vectors.shape)
print("Time delta    :", time_delta_vectors.shape)


# ------------------------------------------------------------
# 3. Concatenate along the feature dimension
# ------------------------------------------------------------

interaction_vectors = torch.cat(
    [
        question_vectors,
        skill_vectors,
        correctness_vectors,
        elapsed_time_vectors,
        attempt_vectors,
        time_delta_vectors,
    ],
    dim=1
)


# ------------------------------------------------------------
# 4. Calculate expected dimension
# ------------------------------------------------------------

expected_dimension = (
    64 + 32 + 8 + 16 + 8 + 8
)


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

assert expected_dimension == 136

assert interaction_vectors.shape == (
    len(processed),
    expected_dimension
)

assert torch.isfinite(
    interaction_vectors
).all()


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("COMBINED INTERACTION REPRESENTATION")
print("=" * 60)

print(
    "\nExpected dimension:",
    expected_dimension
)

print(
    "Actual shape:",
    interaction_vectors.shape
)

print(
    "\nFirst interaction vector shape:",
    interaction_vectors[0].shape
)

print("\nFirst 136-D interaction vector:")
print(interaction_vectors[0])


# ------------------------------------------------------------
# 7. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ All six feature representations combined successfully.")
print(
    f"✓ {len(processed)} interactions → "
    f"{expected_dimension}-D interaction vectors"
)
print("=" * 60)

Individual feature shapes:
Question      : torch.Size([3368, 64])
Skill         : torch.Size([3368, 32])
Correctness   : torch.Size([3368, 8])
Elapsed time  : torch.Size([3368, 16])
Attempt       : torch.Size([3368, 8])
Time delta    : torch.Size([3368, 8])

COMBINED INTERACTION REPRESENTATION

Expected dimension: 136
Actual shape: torch.Size([3368, 136])

First interaction vector shape: torch.Size([136])

First 136-D interaction vector:
tensor([-0.2151, -0.7868, -0.9192,  0.1661, -0.3877, -0.7923,  0.4576, -2.1967,
        -0.6014, -0.3619,  1.5995, -0.3520,  1.2397, -0.5095,  0.8662,  1.4772,
         1.0243, -0.0989,  1.2566, -0.1003, -0.9633,  1.1506,  0.6339, -0.5129,
        -1.0365,  2.1374,  1.3713,  0.4153,  0.8220, -1.4969,  1.1647,  0.1182,
         0.2922,  0.7909,  0.9594, -1.1012, -1.5508,  0.2995,  0.3042,  1.0411,
        -1.3258, -1.1839, -0.4167, -0.3193, -1.1641,  1.0390,  0.6589,  2.0123,
         0.9948,  0.5371,  0.6665,  1.1573,  0.4674,  1.8396,  0.7028, -0.3300

In [10]:
# ============================================================
# LEARNINGDNA ENCODER — STEP 1
# 136-D INTERACTION → 128-D MODEL REPRESENTATION
# ============================================================

import torch
import torch.nn as nn


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

INTERACTION_DIM = 136
MODEL_DIM = 128


# ------------------------------------------------------------
# 2. Validate the input
# ------------------------------------------------------------

if not isinstance(interaction_vectors, torch.Tensor):
    raise TypeError(
        "interaction_vectors must be a PyTorch tensor."
    )

if interaction_vectors.ndim != 2:
    raise ValueError(
        "interaction_vectors must have shape "
        "(number_of_interactions, 136)."
    )

if interaction_vectors.shape[1] != INTERACTION_DIM:
    raise ValueError(
        f"Expected {INTERACTION_DIM} features, "
        f"got {interaction_vectors.shape[1]}."
    )


# ------------------------------------------------------------
# 3. Create the learnable projection
# ------------------------------------------------------------

interaction_projection = nn.Linear(
    in_features=INTERACTION_DIM,
    out_features=MODEL_DIM
)


# ------------------------------------------------------------
# 4. Project 136-D → 128-D
# ------------------------------------------------------------

model_input_vectors = interaction_projection(
    interaction_vectors.float()
)


# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

assert model_input_vectors.shape == (
    len(processed),
    MODEL_DIM
)

assert torch.isfinite(
    model_input_vectors
).all()


# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 60)
print("136-D → 128-D PROJECTION")
print("=" * 60)

print("\nInput shape:")
print(interaction_vectors.shape)

print("\nProjection layer:")
print(interaction_projection)

print("\nOutput shape:")
print(model_input_vectors.shape)

print("\nFirst interaction:")
print("Input:", interaction_vectors[0].shape)
print("Output:", model_input_vectors[0].shape)

print("\nFirst 128-D model representation:")
print(model_input_vectors[0])


# ------------------------------------------------------------
# 7. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ 136-D interaction representation projected to 128-D.")
print(
    f"✓ {len(processed)} interactions → "
    f"{MODEL_DIM}-D model representations"
)
print("=" * 60)

136-D → 128-D PROJECTION

Input shape:
torch.Size([3368, 136])

Projection layer:
Linear(in_features=136, out_features=128, bias=True)

Output shape:
torch.Size([3368, 128])

First interaction:
Input: torch.Size([136])
Output: torch.Size([128])

First 128-D model representation:
tensor([ 1.6580e-01,  1.0703e+00,  6.2556e-01,  3.9445e-01,  1.5988e+00,
         4.0396e-01, -1.1207e+00, -1.1534e-02, -9.6816e-01,  5.6321e-01,
         1.5390e-01, -3.3167e-01,  2.4037e-01,  7.4729e-01,  9.6257e-02,
        -4.7833e-01,  7.3657e-01,  7.9558e-01, -7.3000e-02,  3.8041e-02,
         2.9977e-01,  4.7390e-02,  2.1937e-01,  3.3802e-01, -1.2661e+00,
         3.9049e-01,  7.7803e-01, -2.4316e-01,  1.1340e-01, -2.5100e-01,
         2.3974e-03,  1.4883e+00,  9.6620e-01, -6.5294e-02, -2.0310e-03,
         8.0073e-01,  1.1958e-01,  7.4092e-01, -6.4808e-01, -5.3538e-01,
        -1.2248e+00,  1.4925e-01, -5.1956e-01, -5.6029e-01, -4.1791e-01,
        -1.3435e+00, -5.8720e-01, -6.4217e-01,  3.5557e-01, -4.